# Python Lesson 54: Hypothesis Testing and A/B Testing

**Concept page:** `wiki/concepts/hypothesis-testing-ab-test.md` · **Area:** statistics

**You will be able to:**
- implement the Welch t-test and the two-proportion z-test
- compute confidence intervals
- compare with SciPy

*How to use:* run the cells top to bottom, read the output, then try the **Exercises** in the empty cells before opening the **Solutions** section at the bottom.

In [1]:
import numpy as np
import sympy as sp
import matplotlib
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

## 1. Continuous metric (simulated data shaped like the lab)

In [2]:
rng=np.random.default_rng(42)
ctrl=rng.gamma(4,8.2,2069); var=rng.gamma(4,8.5,2117)
def metrics(x): return len(x),x.mean(),x.std(ddof=1)
(n1,m1,s1),(n2,m2,s2)=metrics(ctrl),metrics(var)
t=(m1-m2)/np.sqrt(s1**2/n1+s2**2/n2)
df=(s1**2/n1+s2**2/n2)**2/((s1**2/n1)**2/(n1-1)+(s2**2/n2)**2/(n2-1))
print(t,df)
from scipy import stats
print(stats.ttest_ind(ctrl,var,equal_var=False))
crit=stats.t.ppf(0.975,df); print("reject H0" if abs(t)>crit else "fail to reject", crit)

-2.3523896429840447 4183.9999965412


TtestResult(statistic=np.float64(-2.3523896429840447), pvalue=np.float64(0.018699302319416647), df=np.float64(4183.9999965412))
reject H0 1.9605311322388208


## 2. Proportions (conversion rate)

In [3]:
x1,n1p,x2,n2p=200,2000,250,2000
p1,p2=x1/n1p,x2/n2p; pp=(x1+x2)/(n1p+n2p)
z=(p1-p2)/np.sqrt(pp*(1-pp)*(1/n1p+1/n2p)); print(z, 2*(1-stats.norm.cdf(abs(z))))
se=np.sqrt(p2*(1-p2)/n2p); print("95% CI for p2:",p2-1.96*se,p2+1.96*se)

-2.501955416802671 0.012350947711191385
95% CI for p2: 0.11050560453140594 0.13949439546859405


## 3. How often does a true null get rejected? (≈ α)

In [4]:
rej=0
for _ in range(2000):
    a=rng.normal(0,1,50); b=rng.normal(0,1,50); rej+=stats.ttest_ind(a,b,equal_var=False).pvalue<0.05
print(rej/2000)

0.0505


## Cambodian textbook corner (កម្មវិធីសិក្សាកម្ពុជា)
Grade 9 statistics (mean) and Grade 12 advanced probability (binomial) are the school-level ingredients; the test statistic standardises a difference of means.

In [5]:
print(m1-m2)

-1.207394772840459


## Exercises

**Exercise 1.** Run the t-test when the variation truly has mean +0.5 SD; estimate the power over 1000 repetitions (~0.7 for n=50).

In [6]:
# your code here

**Exercise 2.** Verify the pooled z-test equals scipy's chi-square test on the 2x2 table (z^2 = chi2, no correction).

In [7]:
# your code here

## Solutions
*(Try the exercises first!)*

In [8]:
# Solution 1
rej=0
for _ in range(1000):
    a=rng.normal(0,1,50); b=rng.normal(0.5,1,50); rej+=stats.ttest_ind(a,b,equal_var=False).pvalue<0.05
print(rej/1000); assert 0.6<rej/1000<0.8

0.711


In [9]:
# Solution 2
tab=np.array([[x1,n1p-x1],[x2,n2p-x2]]); chi=stats.chi2_contingency(tab,correction=False)[0]; print(z**2,chi); assert abs(z**2-chi)<1e-6

6.259780907668228 6.259780907668231
